# PixiJoy Flow (M4) — Colab GPU (quality)

Runtime → **GPU** → **Run all**.

- Auto-fetches **real** training data (no tensorflow_datasets).
- `USE_DRIVE = False` by default (no Google Drive prompt). Set True only to resume after disconnect.
- Pin TFLite only if verify prints **OK**.


In [ ]:
!nvidia-smi || true
import tensorflow as tf
print("TF", tf.__version__, "GPUs", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices('GPU'), 'Select Runtime → GPU before Run all'


In [ ]:
USE_DRIVE = False  # optional resume only

import os, urllib.request, zipfile, shutil
from pathlib import Path
from google.colab import files

ROOT = Path('/content/pixijoy_flow')
DRIVE = Path('/content/drive/MyDrive/pixijoy_ml/flow')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    (DRIVE / 'dist').mkdir(parents=True, exist_ok=True)

ZIP_URL = 'https://raw.githubusercontent.com/Saikumar-S0906/Release-/main/flow_lab.zip'
ok = False
try:
    urllib.request.urlretrieve(ZIP_URL, 'flow_lab.zip')
    with zipfile.ZipFile('flow_lab.zip') as z:
        z.extractall('.')
    ok = Path('train.py').is_file() and Path('prepare_quality_data.py').is_file()
    print('lab zip OK', ok)
except Exception as e:
    print('zip fetch failed:', e)
if not ok:
    print('Upload flow_lab.zip')
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(name) as z:
        z.extractall('.')
    ok = Path('train.py').is_file()
assert ok, 'lab zip missing train.py / prepare_quality_data.py — refresh Runtime and re-fetch'
Path('dist').mkdir(parents=True, exist_ok=True)
if USE_DRIVE:
    d = Path('dist')
    if d.exists() and not d.is_symlink():
        shutil.rmtree(d) if d.is_dir() else d.unlink()
    if not Path('dist').exists():
        Path('dist').symlink_to(DRIVE / 'dist')


In [ ]:
print('prepare_quality_data (real data, no TFDS)…')
rc = get_ipython().system('python -u prepare_quality_data.py --data ./data')
assert rc == 0, 'prepare_quality_data failed'
assert (Path('data') / 'READY').is_file()


In [ ]:
from pathlib import Path
import hashlib
USE_DRIVE = globals().get('USE_DRIVE', False)
OUT = Path('/content/drive/MyDrive/pixijoy_ml/flow/dist/flow.keras') if USE_DRIVE else Path('dist') / 'flow.keras'
OUT.parent.mkdir(parents=True, exist_ok=True)
print('ckpt exists' if OUT.with_suffix('.ckpt.keras').is_file() else 'fresh', OUT)

cmd = "python -u train.py --quality --out \"" + str(OUT) + "\""
print('RUN', cmd)
rc = get_ipython().system(cmd)
assert rc == 0, 'train failed'
rc = get_ipython().system('python -u export_tflite.py --keras "' + str(OUT) + '" --out "dist/flow.tflite"')
assert rc == 0, 'export failed (refusing untrained is correct)'
rc = get_ipython().system('python -u verify_contract.py --quality')
assert rc == 0, 'verify --quality failed — do NOT pin'
p = Path('dist/flow.tflite')
assert p.is_file(), p
print('READY', p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest())


In [ ]:
from google.colab import files
files.download('dist/flow.tflite')
print('Attach flow.tflite to ml-packs-v1 ONLY if verify OK')
